# White-box probe submission (transformer token probe)

One detection method for every model organism: read hidden-state activations
of the assistant's response tokens at a fixed decoder layer and classify them
with a trained transformer token probe. Per-organism probe weights are bundled
under `submission/whitebox_probe/{qwen,gemma,nemotron}_probe/` (hidden sizes
differ per architecture, so weights cannot be shared — the method, features,
architecture, and training procedure are identical). Writes `submission.csv`
with `index,deceptive,score`.

In [ ]:
import os, sys, json
from pathlib import Path

DATASET_NAME = os.environ["DATASET_NAME"]
LIMIT = int(os.environ["ALETHEIA_LIMIT"]) if os.environ.get("ALETHEIA_LIMIT") else None
NNSIGHT_REMOTE = os.environ.get("NNSIGHT_REMOTE", "1").lower() in {"1", "true", "yes"}
THRESHOLD = float(os.environ.get("SUBMISSION_THRESHOLD", "0.5"))

print(f"dataset = {DATASET_NAME}")
print(f"limit   = {LIMIT}")
print(f"remote  = {NNSIGHT_REMOTE}")
print(f"threshold = {THRESHOLD}")

In [ ]:
import numpy as np
import torch
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, "submission")
import util

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")

In [ ]:
# Load the dataset and pick the matching probe weights by base model
from datasets import load_dataset
ds = load_dataset(DATASET_NAME, split="test")
if LIMIT:
    ds = ds.select(range(LIMIT))
print(f"Loaded {len(ds)} examples")

model_id = ds[0]["model"]
lora = ds[0].get("lora", None)
print(f"model = {model_id}")
print(f"lora  = {lora}")

base_model = None
for family in ("gemma", "qwen", "nemotron"):
    if family in model_id.lower():
        base_model = family
        break
if base_model is None:
    print(f"WARNING: no probe weights for base model {model_id}")
else:
    probe_dir = Path(f"submission/whitebox_probe/{base_model}_probe")
    print(f"base_model = {base_model}")
    print(f"probe_dir  = {probe_dir}")

In [ ]:
if base_model is not None:
    # Load probe config, weights, and standardization moments
    with open(probe_dir / "config.json") as f:
        config = json.load(f)

    feature_mean = torch.load(probe_dir / "feature_mean.pt", map_location=device)
    feature_std = torch.load(probe_dir / "feature_std.pt", map_location=device)

    print(f"hidden_dim = {config['hidden_dim']}")
    print(f"layer      = {config['layer']}")

In [ ]:
if base_model is not None:
    # Transformer token probe definition (must match training)
    import math
    
    def sinusoidal_position_encoding(seq_len, d_model, device=None):
        position = torch.arange(seq_len, dtype=torch.float32, device=device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32, device=device)
                             * (-math.log(10000.0) / d_model))
        enc = torch.zeros(seq_len, d_model, device=device)
        enc[:, 0::2] = torch.sin(position * div_term)
        cc = enc[:, 1::2].shape[1]
        enc[:, 1::2] = torch.cos(position * div_term)[:, :cc]
        return enc
    
    class TransformerTokenProbe(torch.nn.Module):
        def __init__(self, hidden_dim, d_model=128, n_heads=4, dim_feedforward=256, n_blocks=2, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.projection = torch.nn.Linear(hidden_dim, d_model)
            block = torch.nn.TransformerEncoderLayer(
                d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward,
                dropout=dropout, batch_first=True)
            self.encoder = torch.nn.TransformerEncoder(block, num_layers=n_blocks)
            self.head = torch.nn.Sequential(torch.nn.Dropout(dropout), torch.nn.Linear(d_model, 1))
        def forward(self, padded_tokens, padding_mask):
            seq_len = padded_tokens.shape[1]
            pe = sinusoidal_position_encoding(seq_len, self.d_model, device=padded_tokens.device)
            x = self.projection(padded_tokens) + pe.unsqueeze(0)
            x = self.encoder(x, src_key_padding_mask=~padding_mask)
            m = padding_mask.unsqueeze(-1).to(x.dtype)
            pooled = (x * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)
            return self.head(pooled).squeeze(-1)
    
    probe = TransformerTokenProbe(
        hidden_dim=config['hidden_dim'],
        d_model=config['d_model'],
        n_heads=config['n_heads'],
        dim_feedforward=config['dim_feedforward'],
        n_blocks=config['n_blocks'],
        dropout=config['dropout'],
    ).to(device)
    probe.load_state_dict(torch.load(probe_dir / "model.pt", map_location=device))
    probe.eval()
    print("Probe loaded and in eval mode.")

In [ ]:
if base_model is not None:
    # Build the nnsight model handle: config/tokenizer load locally, the
    # weights stay on NDIF when tracing remotely
    model = util.build_model(model_id, lora)
    tokenizer = model.tokenizer
    print(f"Model loaded: {type(model).__name__}")

In [ ]:
if base_model is not None:
    # Locate the probed decoder layer; batch sizing comes from the probe config
    # (large models with little deployment headroom need smaller traces)
    layer_modules = util.decoder_layers(model)
    layer_idx = min(config['layer'], len(layer_modules) - 1)
    print(f"Decoder layers: {len(layer_modules)}, using layer {layer_idx}")

    PAD_ID = (tokenizer.pad_token_id if tokenizer.pad_token_id is not None
              else tokenizer.eos_token_id)
    BATCH_TOKEN_BUDGET = config.get("extract_token_budget", 2560)
    MAX_BATCH_ROWS = config.get("extract_max_batch", 32)
    print(f"extraction batches: token budget {BATCH_TOKEN_BUDGET}, "
          f"max {MAX_BATCH_ROWS} rows")

In [ ]:
if base_model is not None:
    # Tokenize everything, compute response spans, build batches
    token_lists, spans, indices = [], [], []
    for i, example in enumerate(ds):
        token_ids, span = util.chat_preprocess(example["messages"], tokenizer, max_len=512)
        token_lists.append(token_ids)
        spans.append(span)
        indices.append(example.get("index", i))

    # Length-sorted batch packing under the token budget and row cap
    lengths = [len(t) for t in token_lists]
    order = sorted(range(len(lengths)), key=lambda p: lengths[p])
    batches, current = [], []
    for pos in order:
        if current and ((len(current) + 1) * lengths[pos] > BATCH_TOKEN_BUDGET
                        or len(current) >= MAX_BATCH_ROWS):
            batches.append(current); current = []
        current.append(pos)
    if current: batches.append(current)
    print(f"{len(token_lists)} examples, {len(batches)} batches")

In [ ]:
if base_model is not None:
    # Extract the probed layer's activations for every response token, all
    # batches bundled into one NDIF session (only values flowing into a final
    # .save() survive a remote session, and captured objects must cloudpickle).
    # NDIF results occasionally download corrupted (EOFError "Ran out of input")
    # or a remote session drops mid-run; the organizers advise retrying these
    # transient failures, so the whole session is wrapped in a bounded retry.
    import time
    from contextlib import nullcontext

    def extract_activations():
        session = model.session(remote=True) if NNSIGHT_REMOTE else nullcontext()
        with session:
            pieces = []
            for batch_positions in batches:
                batch_tokens = [token_lists[p] for p in batch_positions]
                batch_spans = [spans[p] for p in batch_positions]
                width = max(len(t) for t in batch_tokens)
                rows = len(batch_tokens)
                input_ids = torch.full((rows, width), PAD_ID, dtype=torch.long)
                attn_mask = torch.zeros(rows, width, dtype=torch.long)
                resp_mask = torch.zeros(rows, width, dtype=torch.bool)
                for row, (tokens, (start, end)) in enumerate(zip(batch_tokens, batch_spans)):
                    input_ids[row, :len(tokens)] = torch.tensor(tokens)
                    attn_mask[row, :len(tokens)] = 1
                    resp_mask[row, start:end] = True

                with model.trace({"input_ids": input_ids, "attention_mask": attn_mask}) as tracer:
                    hidden = layer_modules[layer_idx].output
                    if isinstance(hidden, tuple):
                        hidden = hidden[0]
                    mask_bool = resp_mask.to(hidden.device)
                    selected = hidden[mask_bool].to(torch.float16).detach().cpu().save()
                    tracer.stop()
                pieces.append(selected)

            flat = torch.cat(pieces, dim=0)
            if NNSIGHT_REMOTE:
                flat = flat.save()
        # Guard non-finite values once the tensor is back on the client.
        finfo = torch.finfo(torch.float16)
        return flat.cpu().clamp(finfo.min, finfo.max)

    def is_transient(err):
        # EOFError is the corrupt-NDIF-download failure the organizers flagged;
        # the string markers catch dropped/streamed session transport errors.
        if isinstance(err, EOFError):
            return True
        markers = ("ran out of input", "eof", "connection", "reset", "timed out",
                   "timeout", "corrupt", "temporarily", "502", "503", "504")
        return any(m in str(err).lower() for m in markers)

    MAX_ATTEMPTS = int(os.environ.get("EXTRACT_MAX_ATTEMPTS", "4"))
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            flat_batch = extract_activations()
            break
        except Exception as err:
            if attempt >= MAX_ATTEMPTS or not is_transient(err):
                raise
            wait = min(30, 2 ** attempt)
            print(f"transient extraction error on attempt {attempt}/{MAX_ATTEMPTS}: "
                  f"{type(err).__name__}: {err}; retrying in {wait}s")
            time.sleep(wait)

    # Tokens arrive in batch-traversal order (batches are length-sorted); reorder
    # back to dataset order for scoring.
    span_lengths = [end - start for start, end in spans]
    batch_order = [p for batch in batches for p in batch]
    piece_lengths = [span_lengths[p] for p in batch_order]
    piece_offsets = np.cumsum([0] + piece_lengths).astype(np.int64)
    slot_of = {p: slot for slot, p in enumerate(batch_order)}
    flat_features = torch.cat([
        flat_batch[piece_offsets[slot_of[p]]:piece_offsets[slot_of[p]] + span_lengths[p]]
        for p in range(len(spans))]).to(device)
    offsets = np.cumsum([0] + span_lengths).astype(np.int64)
    print(f"Extracted {flat_features.shape[0]} tokens, shape={tuple(flat_features.shape)}")

In [ ]:
if base_model is not None:
    # Score all examples
    def score_examples(flat_features, offsets, probe, feature_mean, feature_std, token_budget=8192):
        N = len(offsets) - 1
        lengths = (offsets[1:] - offsets[:-1]).tolist()
        order = sorted(range(N), key=lambda p: lengths[p])
        batches, current = [], []
        for pos in order:
            w = lengths[pos]
            if current and (len(current) + 1) * max(lengths[p] for p in current + [pos]) > token_budget:
                batches.append(current); current = []
            current.append(pos)
        if current: batches.append(current)
        
        scores = np.zeros(N, dtype=np.float64)
        with torch.no_grad():
            for row_ids in batches:
                ml = max(lengths[r] for r in row_ids)
                h = flat_features.shape[1]
                padded = torch.zeros(len(row_ids), ml, h, dtype=torch.float32, device=device)
                mask = torch.zeros(len(row_ids), ml, dtype=torch.bool, device=device)
                for pos, row in enumerate(row_ids):
                    s, e = int(offsets[row]), int(offsets[row+1])
                    padded[pos, :e-s] = flat_features[s:e].to(torch.float32)
                    mask[pos, :e-s] = True
                x = (padded - feature_mean) / feature_std
                x = x * mask.unsqueeze(-1)
                logits = probe(x, mask)
                for pos, row in enumerate(row_ids):
                    scores[row] = torch.sigmoid(logits[pos]).item()
        return scores
    
    scores = score_examples(flat_features, offsets, probe, feature_mean, feature_std)
    print(f"Scored {len(scores)} examples, range [{scores.min():.4f}, {scores.max():.4f}]")


In [ ]:
if base_model is not None:
    # Write submission.csv; a probe may carry its own calibrated threshold
    util.write_submission(indices, scores,
                          threshold=config.get("threshold", THRESHOLD))
else:
    # No probe weights for this base model: emit uninformative default scores
    indices = [example.get("index", i) for i, example in enumerate(ds)]
    scores = np.full(len(indices), 0.5)
    util.write_submission(indices, scores, threshold=THRESHOLD)
print("Done.")